In [1]:
import pickle
import os ,re
import numpy as np
import torch
from torch.utils.data import DataLoader
# from avf import MyData
# from dsac import MyData
# from data.newdata import newMyData
from data.data1 import Data
from tqdm import tqdm

In [2]:
path = ['../../data/RL/easydata']
paths = os.listdir(path=path[0])
path_files = list()
for i in paths:
    path_files.append(os.path.join(path[0] , i))
num_split = 50
split_lists = np.array_split(path_files, num_split)
split_lists = [list(sublist) for sublist in split_lists]

In [3]:
# 第一个就是obs所带来的动作，最后一个动作一定是和前一个一样的，因为visual和上一个相等
def bi(batch_labels , model_action):
    batch_labels = np.array(batch_labels)
    bi = 0
    for i in range(len(batch_labels)):
        if batch_labels[i] == model_action[i]:
            bi +=1 
    return bi/len(batch_labels)

In [4]:
dataset = Data(path=split_lists[3])
dataloader = DataLoader(dataset=dataset, batch_size=16, shuffle=True)

deal batch_data: 100%|██████████| 14/14 [00:00<00:00, 19612.64batch_datas/s]


In [5]:

from dsac import AVNet
model_path = './checkpoint/DSAC_easy_100000.pth'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
agent = AVNet(128 , 4 ,width_dim=128 , height_dim=36).to(device)
agent.load_state_dict(torch.load(model_path))
agent.eval()

AVNet(
  (avf): AVFNet(
    (audio): Sequential(
      (0): Conv2d(2, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU()
      (2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU()
      (4): Flatten(start_dim=1, end_dim=-1)
      (5): Linear(in_features=294912, out_features=128, bias=True)
      (6): ReLU()
      (7): Linear(in_features=128, out_features=64, bias=True)
    )
    (visual): Sequential(
      (0): Conv2d(4, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU()
      (2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU()
      (4): Flatten(start_dim=1, end_dim=-1)
      (5): Linear(in_features=1048576, out_features=128, bias=True)
      (6): ReLU()
      (7): Linear(in_features=128, out_features=64, bias=True)
    )
    (mask): Sequential(
      (0): Linear(in_features=128, out_features=128, bias=True)
      (1): ReLU()
      (2): Linear(in_features=128, out_featu

In [9]:
batch_data = next(iter(dataloader))
batch_pre_audio, batch_pre_visual, batch_next_audio , batch_next_visual,batch_done, batch_reward ,batch_labels = batch_data
batch_next_visual = batch_next_visual.to(device).float()
batch_next_audio = batch_next_audio.to(device).float()
batch_pre_audio = batch_pre_audio.to(device).float()
batch_pre_visual = batch_pre_visual.to(device).float()
model_action = agent(batch_pre_audio,batch_pre_visual)[0].max(-1)[1].tolist()
# 可以看到每一次都是策略训练废掉了qq
# 策略还是废掉了

In [10]:
model_action

[0, 2, 3, 3, 1, 1, 0, 2, 1, 0, 2, 0, 0, 1, 1, 0]

In [11]:
batch_labels

tensor([0, 2, 3, 3, 1, 1, 0, 2, 1, 0, 2, 0, 0, 1, 1, 0])

In [ ]:
batch_reward

tensor([10.0000, 10.0000,  2.1124,  2.3936, 10.0000,  2.4204, 10.0000,  2.4836,
         2.4333,  2.4230, 10.0000,  2.4621, 10.0000,  2.4448,  2.4974, 10.0000])

In [12]:
bi_list = list()
for i in range(len(split_lists)):
    dataset = Data(path=split_lists[i])
    dataloader = DataLoader(dataset=dataset, batch_size=16, shuffle=True)
    for batch_data in tqdm(dataloader ,desc='dataloder'):
        batch_pre_audio, batch_pre_visual, batch_next_audio , batch_next_visual,batch_done, batch_reward ,batch_labels = batch_data
        batch_next_visual = batch_next_visual.to(device).float()
        batch_next_audio = batch_next_audio.to(device).float()
        batch_pre_audio = batch_pre_audio.to(device).float()
        batch_pre_visual = batch_pre_visual.to(device).float()
        # model_action = agent(batch_pre_audio,batch_pre_visual).max(-1)[1].tolist()
        model_action = agent(batch_pre_audio,batch_pre_visual)[0].max(-1)[1].tolist()
        bi_list.append(bi(batch_labels=batch_labels , model_action=model_action))

dataloder: 100%|██████████| 8/8 [00:01<00:00,  6.38it/s]


In [13]:
print(f"max_bi : {max(bi_list)},\nmin_bi : {min(bi_list)}, \nmean_bi :{np.array(bi_list).mean()}")

max_bi : 1.0,
min_bi : 0.0, 
mean_bi :0.9494978509535471


In [16]:
a  = torch.zeros((61,2,18000))

In [17]:
torch.all(a == 0)

tensor(True)

In [5]:
# path = '../data/RL/newdone'
def load(path):
    files = os.listdir(path)
    files_path = list()
    for file in files :
        files_path.append(os.path.join(path, file))
    return files_path

In [4]:
files = load(path)

In [5]:
mydata = MyData(path)

FileNotFoundError: [Errno 2] No such file or directory: 'd'

In [4]:
with open(files[0],'rb') as f:
    data = pickle.load(f)

NameError: name 'files' is not defined

In [28]:
preaudio,previsual, nextaudio, nextvisua,done,reward,action = mydata.__getitem__(0)

In [ ]:
np.array_equal(data[0][0]['camera'][1]  , nextvisua)

True

: 

In [1]:
# test dataset
from avn import MyData
path = ['../data/RL/newdone']

In [2]:
mydata = MyData(path)

load files:   0%|          | 0/700 [00:00<?, ?file/s]

deal batch_data: 100%|██████████| 700/700 [00:00<00:00, 49341.44batch_datas/s]

不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等
不相等


In [6]:
files = load(path[0])

In [7]:
files[0]

'../data/RL/newdone/rl_episode_699.pkl'

In [10]:
with open('../data/RL/newdone/rl_episode_699.pkl', 'rb') as f:
    data = pickle.load(f)

In [9]:
data[0][0].keys()

dict_keys(['camera', 'audio', 'step', 'rl_pred', 'rl_logits', 'rl_value', 'reward', 'mask', 'lstm_h', 'lstm_c'])

In [14]:
data[0][0]['reward']

array([  0.       ,  10.       ,   2.4662066,   2.4586916,   2.4483824,
         2.4337363,   2.4119735,  10.       ,   2.416885 ,  10.       ,
         2.4104476,  10.       ,   2.4174547,   2.3424149,  10.       ,
       100.       , 100.       ,   0.       ,   0.       ,   0.       ,
         0.       ,   0.       ,   0.       ,   0.       ,   0.       ,
         0.       ,   0.       ,   0.       ,   0.       ,   0.       ,
         0.       ,   0.       ,   0.       ,   0.       ,   0.       ,
         0.       ,   0.       ,   0.       ,   0.       ,   0.       ,
         0.       ,   0.       ,   0.       ,   0.       ,   0.       ,
         0.       ,   0.       ,   0.       ,   0.       ,   0.       ,
         0.       ,   0.       ,   0.       ,   0.       ,   0.       ,
         0.       ,   0.       ,   0.       ,   0.       ,   0.       ,
         0.       ], dtype=float32)

In [15]:
data[0][0]['step']

array([ 0,  0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15,
       16,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
        0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
        0,  0,  0,  0,  0,  0,  0,  0,  0,  0])

In [47]:
data[0][0]['rl_pred']

array([0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 1, 0, 2,
       0, 1, 0, 2, 0, 0, 2, 2, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 2,
       0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [16]:
len(data[2])

16

In [3]:
mydata.__len__()

24631
24631
24631
24631
24553
24631
24631


24631

In [12]:
len(data[2])

16

In [6]:
# 核心查看reward， done ， state
preaudio,previsual, nextaudio, nextvisual,done,reward,action   = mydata.__getitem__(24552)

In [7]:
action

1

In [8]:
done

1

In [24]:
for i in range(1000):
    # print(i)
    preaudio,previsual, nextaudio, nextvisual,done,reward,action   = mydata.__getitem__(672+i)

In [25]:
mydata.__len__()

24631

In [19]:
done

0

In [9]:
action

2

In [40]:
data[0][0]['audio'][1]

array([[ 3.75192100e-09,  2.66960276e-09,  7.51610330e-10, ...,
        -6.12057652e-03, -5.28840907e-03,  1.39715587e-04],
       [ 1.42392553e-09, -1.46208753e-10, -2.45285459e-09, ...,
         6.69605145e-03,  7.13925110e-03,  1.08815385e-02]], dtype=float32)

In [41]:
data[0][0]['camera'][1]

array([[[149., 146., 129., 255.],
        [149., 147., 129., 255.],
        [149., 148., 130., 255.],
        ...,
        [143., 144., 128., 255.],
        [143., 144., 128., 255.],
        [143., 144., 128., 255.]],

       [[148., 147., 127., 255.],
        [148., 147., 127., 255.],
        [148., 147., 127., 255.],
        ...,
        [143., 144., 128., 255.],
        [143., 144., 128., 255.],
        [141., 142., 126., 255.]],

       [[150., 146., 127., 255.],
        [148., 147., 127., 255.],
        [148., 147., 127., 255.],
        ...,
        [143., 144., 127., 255.],
        [141., 142., 126., 255.],
        [141., 142., 128., 255.]],

       ...,

       [[ 60.,  55.,  45., 255.],
        [ 59.,  54.,  45., 255.],
        [ 60.,  55.,  45., 255.],
        ...,
        [107., 104.,  86., 255.],
        [110., 107.,  91., 255.],
        [110., 107.,  91., 255.]],

       [[ 72.,  63.,  51., 255.],
        [ 72.,  63.,  51., 255.],
        [ 76.,  68.,  55., 255.],
        .

In [16]:
print(np.array_equal(np.array(nextaudio) , data[0][0]['audio'][17]))

False


In [18]:
print(np.array_equal(np.array(nextvisual) , data[0][0]['camera'][17]))

True


In [19]:
reward

100.0

In [24]:
done

0

In [1]:
import torch
a = torch.Tensor([[1,2,3,4],[5,6,7,8]])
b = torch.Tensor([[2,1,4,3],[6,5,8,7]])

In [2]:
a

tensor([[1., 2., 3., 4.],
        [5., 6., 7., 8.]])

In [3]:
b

tensor([[2., 1., 4., 3.],
        [6., 5., 8., 7.]])

In [4]:
torch.min(a,b)

tensor([[1., 1., 3., 3.],
        [5., 5., 7., 7.]])